# Global Terrorism Data Analysis

**Interactive data-analytics project using the GTTAC Record of Incident Database (GRID).**

This notebook analyzes a supplied snapshot of the **Global Terrorism Trends and Analysis Center (GTTAC) Record of Incident Database (GRID)**. GRID is an incident-level, open-source database. The Incident File contains one row per unique incident; the Perpetrator File can contain multiple rows for the same incident because an incident may involve multiple perpetrators; the Source File can contain multiple source records for an incident.

The notebook has been adapted from the original synthetic-data framework to use the actual GRID structure. It does **not** generate synthetic incidents and it does not invent synthetic-only variables such as Success or PropertyDamage.

- **Data loading** - Incident, Perpetrator and Source GRID files.
- **Cleaning & preprocessing** - dates, duplicates, missing geographic fields and derived analytical fields.
- **Statistical analysis** - distribution tests, group comparisons, chi-square association and point-biserial correlation.
- **Exploratory data analysis (EDA)** - temporal, geographic, tactic, weapon, victim and casualty views.
- **Interactive dashboard** - a self-contained Plotly HTML dashboard generated from the GRID snapshot.

### Important data-use note
The raw GRID files should **not** be committed to this public repository. Keep them in the Colab runtime or another permitted private location and upload them when running the notebook. The dashboard and every visual using GRID data include source attribution as required by the supplied GTTAC Terms of Use and Citation Policy.


## Abstract

Terrorism incident data is multidimensional: time, place, tactics, weapons, victims, perpetrators and casualties vary across incidents. This project makes those dimensions visible using GRID.

- **Incident frequency** - incidents and casualties by year.
- **Geographical distribution** - country and region concentration using GRID geographic fields.
- **Tactic and weapon analysis** - parent categories coded by GRID, preserving the fact that an incident may have multiple coded categories.
- **Victim analysis** - actual victim parent categories from GRID.
- **Casualty statistics** - fatalities, injuries and hostage/kidnapping/missing counts, including high-casualty outliers.
- **Perpetrator patterns** - perpetrator names aggregated from the GRID Perpetrator File.
- **Interactive dashboard** - trends, rankings, casualty relationships and a geographic incident map.

The analysis period is determined directly from the supplied GRID snapshot rather than being hard-coded to the synthetic notebook's 2010-2024 period.


## Dataset overview

The notebook uses the three supplied GRID files:

| File | Role |
|---|---|
| 20260511-GRID_INCIDENTS.csv | One row per unique incident |
| 20260511-GRID_PERPS.csv | Perpetrator involvement records; an incident can appear multiple times |
| 20260511-GRID_SOURCE.csv | Source/publication records; an incident can appear multiple times |

Core analytical fields are mapped from the GRID schema:

| Analytical field | GRID source |
|---|---|
| IncidentID | unique_incident_id |
| Date / Year / Month | incident_date, incident_year, incident_month |
| Region / Country / City | region_txt, country_genc_txt, city |
| Lat / Lon | latitude, longitude |
| Fatalities / Injuries / Hostages | num_killed, num_wounded, num_hostkid |
| Suicide | is_suicide |
| Claimed | claimed |
| TacticType | GRID tactic parent-category flags |
| WeaponType | GRID weapon parent-category flags |
| TargetType | GRID actual-victim parent-category flags |
| TerroristGroup | perp_name aggregated from the Perpetrator File |

GRID coded parent categories are multi-label fields. The notebook preserves all coded parent categories in the corresponding Type fields and creates a separate PrimaryTactic / PrimaryWeapon analytical field only where a single category is required for a statistical test or comparison.


## Data cleaning & preprocessing

The GRID snapshot is treated as observed incident data, not synthetic data.

Cleaning and preparation include:

- enforce one row per unique_incident_id in the Incident File;
- parse the GRID incident date;
- retain missing city/coordinate values rather than inventing locations;
- report missingness explicitly;
- protect casualty fields from impossible negative values if encountered;
- aggregate perpetrator names by unique_incident_id;
- derive multi-label tactic, weapon and actual-victim parent-category fields from the GRID binary flags;
- derive PrimaryTactic and PrimaryWeapon only as documented analytical simplifications;
- create normalized/log casualty fields for distribution analysis.

No synthetic country centroids, fictional perpetrator names, success flags, or property-damage flags are introduced.


In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_style('whitegrid')

SEED = 42

HAS_SCIPY = True
HAS_PLOTLY = True
try:
    from scipy import stats
except Exception:
    HAS_SCIPY = False
try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    HAS_PLOTLY = False

def show_plotly(fig, h=440, w=None):
    """Render a plotly figure gracefully (needs nbformat for inline display)."""
    if not HAS_PLOTLY:
        print('[Plotly unavailable - skipping figure]')
        return
    layout = dict(height=h)
    if w:
        layout['width'] = w
    fig.update_layout(**layout)
    try:
        fig.show()
    except Exception:
        print('[Inline plotly display needs: pip install nbformat] (run cells in a kernel)')


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('/content')
INCIDENTS_FILE = DATA_DIR / '20260511-GRID_INCIDENTS.csv'
PERPS_FILE = DATA_DIR / '20260511-GRID_PERPS.csv'
SOURCE_FILE = DATA_DIR / '20260511-GRID_SOURCE.csv'

if not INCIDENTS_FILE.exists():
    INCIDENTS_FILE = Path('20260511-GRID_INCIDENTS.csv')
if not PERPS_FILE.exists():
    PERPS_FILE = Path('20260511-GRID_PERPS.csv')
if not SOURCE_FILE.exists():
    SOURCE_FILE = Path('20260511-GRID_SOURCE.csv')

missing = [str(p) for p in [INCIDENTS_FILE, PERPS_FILE, SOURCE_FILE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'GRID input file(s) not found. Upload the three supplied CSV files to the Colab runtime, then rerun this cell:\n'
        + '\n'.join(missing)
    )

incidents_raw = pd.read_csv(INCIDENTS_FILE, low_memory=False)
perps_raw = pd.read_csv(PERPS_FILE, low_memory=False)
source_raw = pd.read_csv(SOURCE_FILE, low_memory=False)

print('GRID Incident File:', incidents_raw.shape)
print('GRID Perpetrator File:', perps_raw.shape)
print('GRID Source File:', source_raw.shape)

TACTIC_LEVEL1 = {
    'tactic_lvl1_as': 'Assault',
    'tactic_lvl1_tr': 'Trauma',
    'tactic_lvl1_cv': 'Covert',
    'tactic_lvl1_co': 'Coordinated',
    'tactic_lvl1_ex': 'Exploitive',
    'tactic_lvl1_un': 'Unknown',
}
WEAPON_LEVEL1 = {
    'weapon_lvl1_ex': 'Explosives',
    'weapon_lvl1_fi': 'Firearms',
    'weapon_lvl1_im': 'Improvised Explosive Device (IED)',
    'weapon_lvl1_in': 'Incendiary',
    'weapon_lvl1_ml': 'Melee',
    'weapon_lvl1_ot': 'Other',
    'weapon_lvl1_ua': 'Unmanned Aerial Vehicle (UAV)',
    'weapon_lvl1_un': 'Unknown',
}
VICTIM_LEVEL1 = {
    'victim_lvl1_ci': 'Groups',
    'victim_lvl1_go': 'Government',
    'victim_lvl1_gp': 'General Population',
    'victim_lvl1_mi': 'Military',
    'victim_lvl1_ot': 'Other',
    'victim_lvl1_po': 'Political',
    'victim_lvl1_pr': 'Professions',
    'victim_lvl1_nn': 'None',
}

def multi_labels(row, mapping):
    labels = []
    for col, label in mapping.items():
        if col in row.index and pd.to_numeric(row[col], errors='coerce') == 1:
            labels.append(label)
    return '; '.join(labels) if labels else 'Unknown/Unclear'

def first_label(row, mapping):
    for col, label in mapping.items():
        if col in row.index and pd.to_numeric(row[col], errors='coerce') == 1:
            return label
    return 'Unknown'

df = incidents_raw.copy()
df['IncidentID'] = df['unique_incident_id']
df['Date'] = pd.to_datetime(df['incident_date'], errors='coerce')
df['Year'] = pd.to_numeric(df['incident_year'], errors='coerce').astype('Int64')
df['Month'] = pd.to_numeric(df['incident_month'], errors='coerce').astype('Int64')
df['Region'] = df['region_txt']
df['Country'] = df['country_genc_txt']
df['City'] = df['city']
df['Lat'] = pd.to_numeric(df['latitude'], errors='coerce')
df['Lon'] = pd.to_numeric(df['longitude'], errors='coerce')
df['Fatalities'] = pd.to_numeric(df['num_killed'], errors='coerce')
df['Injuries'] = pd.to_numeric(df['num_wounded'], errors='coerce')
df['Hostages'] = pd.to_numeric(df['num_hostkid'], errors='coerce')
df['Suicide'] = pd.to_numeric(df['is_suicide'], errors='coerce').fillna(0).astype(int)
df['Claimed'] = pd.to_numeric(df['claimed'], errors='coerce').fillna(0).astype(int)

df['TacticType'] = df.apply(lambda r: multi_labels(r, TACTIC_LEVEL1), axis=1)
df['WeaponType'] = df.apply(lambda r: multi_labels(r, WEAPON_LEVEL1), axis=1)
df['TargetType'] = df.apply(lambda r: multi_labels(r, VICTIM_LEVEL1), axis=1)

# Derived analytical fields: these are not GRID variables.
df['PrimaryTactic'] = df.apply(lambda r: first_label(r, TACTIC_LEVEL1), axis=1)
df['PrimaryWeapon'] = df.apply(lambda r: first_label(r, WEAPON_LEVEL1), axis=1)

perp_names = (
    perps_raw.dropna(subset=['unique_incident_id', 'perp_name'])
    .assign(perp_name=lambda x: x['perp_name'].astype(str).str.strip())
    .loc[lambda x: x['perp_name'].ne('')]
    .groupby('unique_incident_id')['perp_name']
    .agg(lambda s: '; '.join(pd.unique(s)))
    .rename('TerroristGroup')
)
df = df.merge(perp_names, left_on='IncidentID', right_index=True, how='left')
df['TerroristGroup'] = df['TerroristGroup'].fillna('Unknown / Unclaimed')

source_incidents = source_raw['unique_incident_id'].nunique()
source_records = len(source_raw)

print('Analytical incident rows:', len(df))
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Distinct source-linked incidents:', source_incidents)
print('Source records:', source_records)


## Descriptive statistics

First pass over the GRID incident file: dimensions, casualty statistics, missingness, geography, tactics, weapons and perpetrator coverage.


In [ ]:
print('Shape:', df.shape)
print('Incident ID unique:', df['IncidentID'].nunique())
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
print()

print('Casualty statistics:')
print(df[['Fatalities', 'Injuries', 'Hostages', 'Year', 'Suicide', 'Claimed']].describe().to_string())
print()

print('Missing values in key analytical fields:')
key_cols = ['Date', 'Region', 'Country', 'City', 'Lat', 'Lon', 'Fatalities', 'Injuries',
            'Hostages', 'TacticType', 'WeaponType', 'TargetType', 'TerroristGroup']
print(df[key_cols].isna().sum().to_string())
print()

print('Primary tactics:')
print(df['PrimaryTactic'].value_counts().to_string())
print()

print('Primary weapons:')
print(df['PrimaryWeapon'].value_counts().to_string())
print()

print('Regions by incidents:')
print(df['Region'].value_counts().head(15).to_string())
print()

print('Top 10 countries:')
print(df['Country'].value_counts().head(10).to_string())


In [ ]:
before = len(df)

df = df.drop_duplicates(subset=['IncidentID']).reset_index(drop=True)

# Missing coordinates/cities are retained and reported. No synthetic locations are created.
pre_neg_k = int((df['Fatalities'] < 0).sum())
pre_neg_w = int((df['Injuries'] < 0).sum())
pre_neg_h = int((df['Hostages'] < 0).sum())

df['Fatalities'] = df['Fatalities'].clip(0, None)
df['Injuries'] = df['Injuries'].clip(0, None)
df['Hostages'] = df['Hostages'].clip(0, None)

df['FatalNorm'] = (df['Fatalities'] - df['Fatalities'].min()) / max(
    df['Fatalities'].max() - df['Fatalities'].min(), 1
)
df['LogFatal'] = np.log1p(df['Fatalities'])
df['LogInj'] = np.log1p(df['Injuries'])
df['LogHostages'] = np.log1p(df['Hostages'])

q1, q3 = df['Fatalities'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
out_f = int((df['Fatalities'] > upper).sum())

print('Rows before cleaning:', before)
print('Rows after cleaning:', len(df))
print('Duplicate incident IDs removed:', before - len(df))
print('Negative fatalities:', pre_neg_k)
print('Negative injuries:', pre_neg_w)
print('Negative hostages:', pre_neg_h)
print('Missing key values:', int(df[key_cols].isna().sum().sum()))
print('Fatalities IQR upper:', round(upper, 1), '| high outliers:', out_f)
print()
print('Highest-fatality incidents:')
print(df.nlargest(8, 'Fatalities')[['Country', 'Date', 'PrimaryTactic', 'PrimaryWeapon',
                                   'Fatalities', 'Injuries', 'Hostages']].to_string(index=False))
print()
print('Derived columns:', ['FatalNorm', 'LogFatal', 'LogInj', 'LogHostages'])
print('Mean fatalities/incident:', round(df['Fatalities'].mean(), 2))
print('Median fatalities:', int(df['Fatalities'].median()))


## Exploratory data analysis (EDA)

Temporal trends, geographic concentration, tactics, weapons, actual-victim categories, casualties and perpetrator patterns. GRID multi-label parent categories are counted without pretending that each incident has exactly one tactic or weapon.


In [ ]:
pals = sns.color_palette('Set2')

yct = df.groupby('Year')['IncidentID'].count()
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(yct.index, yct.values, marker='o', ms=4, color='#2c7fb8', lw=2)
ax.fill_between(yct.index, yct.values, alpha=0.15, color='#2c7fb8')
ax.set_title('GRID terrorist incidents per year')
ax.set_xlabel('Year'); ax.set_ylabel('Incidents')
plt.tight_layout(); plt.show()

t15 = df['Country'].value_counts().head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(t15.index, t15.values, color=pals)
ax.set_title('Top 15 countries by number of GRID incidents')
ax.set_xlabel('Incidents')
plt.tight_layout(); plt.show()

at = df['PrimaryTactic'].value_counts()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(at.index, at.values, color=pals)
ax.set_title('Incidents by primary tactic category')
ax.set_ylabel('Incidents')
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()

reg = df['Region'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(reg.values, labels=reg.index, autopct='%.1f%%', colors=pals,
       startangle=90, textprops={'fontsize': 8})
ax.set_title('Incidents by GRID region')
plt.tight_layout(); plt.show()

wkr = df['PrimaryWeapon'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(wkr.values, labels=wkr.index, autopct='%.1f%%', colors=pals,
       startangle=90, textprops={'fontsize': 8})
ax.set_title('Incidents by primary weapon category')
plt.tight_layout(); plt.show()

yf = df.groupby('Year')['Fatalities'].sum()
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(yf.index, yf.values, marker='s', ms=4, color='#a50f15', lw=2)
ax.fill_between(yf.index, yf.values, alpha=0.12, color='#a50f15')
ax.set_title('Total fatalities per year')
ax.set_xlabel('Year'); ax.set_ylabel('Fatalities')
plt.tight_layout(); plt.show()


In [ ]:
pals = sns.color_palette('Set2')

victim_counts = {}
for col, label in VICTIM_LEVEL1.items():
    victim_counts[label] = int(pd.to_numeric(incidents_raw[col], errors='coerce').fillna(0).sum())
tg = pd.Series(victim_counts).sort_values().tail(10)
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(tg.index, tg.values, color=pals)
ax.set_title('Actual victim parent categories in GRID')
ax.set_xlabel('Incidents with category coded')
plt.tight_layout(); plt.show()

gr = perps_raw.groupby('perp_name')['unique_incident_id'].nunique().sort_values().tail(10)
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(gr.index, gr.values, color=pals)
ax.set_title('Top perpetrator names by distinct incidents')
ax.set_xlabel('Distinct incidents')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(df['LogFatal'], bins=30, color='#2c7fb8', edgecolor='white')
axes[0].set_title('Distribution of log1p fatalities')
axes[0].set_xlabel('log1p(fatalities)')
topw = df['PrimaryWeapon'].value_counts().head(5).index
axes[1].boxplot([df.loc[df['PrimaryWeapon'] == w, 'LogFatal'].values for w in topw])
axes[1].set_xticklabels(topw, rotation=15, ha='right')
axes[1].set_title('Fatalities by primary weapon (log1p)')
axes[1].set_ylabel('log1p(fatalities)')
plt.tight_layout(); plt.show()

s1 = df.loc[df['Suicide'] == 1, 'Fatalities'].values
s0 = df.loc[df['Suicide'] == 0, 'Fatalities'].values
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].boxplot([s0, s1], showfliers=False)
axes[0].set_xticklabels(['Non-suicide', 'Suicide'])
axes[0].set_title('Fatalities distribution')
axes[0].set_ylabel('Fatalities')
axes[1].bar(['Non-suicide', 'Suicide'], [s0.mean(), s1.mean()], color=pals)
axes[1].set_title('Mean fatalities per incident')
axes[1].set_ylabel('Mean fatalities')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.scatter(df['Injuries'], df['Fatalities'], s=8, alpha=0.35, color='#2c7fb8')
ax.set_title('Injuries vs fatalities per incident')
ax.set_xlabel('Injuries'); ax.set_ylabel('Fatalities')
plt.tight_layout(); plt.show()

ct = pd.crosstab(df['PrimaryTactic'], df['Region'])
ctp = (ct / ct.sum(0) * 100)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(ctp, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=.5, ax=ax)
ax.set_title('Primary tactic mix by region (%, column-normalized)')
plt.tight_layout(); plt.show()


## Statistical analysis

- **D'Agostino-Pearson test**: distribution of fatalities and log-transformed fatalities.
- **Welch t-test**: suicide vs. non-suicide incidents using log-transformed fatalities.
- **ANOVA**: exploratory comparison of log fatalities across the most common primary weapon categories.
- **Chi-square**: primary tactic vs. region.
- **Point-biserial correlation**: suicide indicator vs. fatalities.
- **Correlation heatmap**: casualties and selected binary indicators.

The original synthetic-only Success and PropertyDamage analyses have been removed because those variables do not exist in the GRID Incident File.


In [ ]:
if HAS_SCIPY:
    f = df['Fatalities'].values
    sh, p = stats.normaltest(f)
    print("1) D'Agostino-Pearson on Fatalities: stat=%.2f  p=%.4g  -> %s" %
          (sh, p, 'NOT normal' if p < 0.05 else 'approx normal'))
    sh2, p2 = stats.normaltest(df['LogFatal'].values)
    print("   D'Agostino-Pearson on LogFatal: stat=%.2f  p=%.4g  -> %s" %
          (sh2, p2, 'NOT normal' if p2 < 0.05 else 'approx normal'))

    suicide_f = df.loc[df['Suicide'] == 1, 'LogFatal'].values
    nonsuicide_f = df.loc[df['Suicide'] == 0, 'LogFatal'].values
    t, pt = stats.ttest_ind(suicide_f, nonsuicide_f, equal_var=False)
    print()
    print('2) Welch t-test on log1p fatalities (suicide vs non-suicide): t=%.2f  p=%.4g  -> %s' %
          (t, pt, 'significant difference' if pt < 0.05 else 'no difference'))

    common_weapons = df['PrimaryWeapon'].value_counts().head(5).index
    wg = [df.loc[df['PrimaryWeapon'] == w, 'LogFatal'].values for w in common_weapons]
    F, pF = stats.f_oneway(*wg)
    print()
    print('3) ANOVA - log fatalities across common primary weapons: F=%.2f  p=%.4g  -> %s' %
          (F, pF, 'significant difference' if pF < 0.05 else 'no difference'))

    ct = pd.crosstab(df['PrimaryTactic'], df['Region'])
    chi, pchi, ddof, _ = stats.chi2_contingency(ct)
    print()
    print('4) Chi-square (primary tactic x region): chi2=%.2f  dof=%d  p=%.4g  -> %s' %
          (chi, ddof, pchi, 'association' if pchi < 0.05 else 'independent'))

    pb = stats.pointbiserialr(df['Suicide'], df['Fatalities'])
    print()
    print('5) Point-biserial (Suicide vs Fatalities): r=%.3f  p=%.4g' % (pb[0], pb[1]))
else:
    print('[SciPy unavailable - statistics skipped]')

corr_cols = ['Fatalities', 'Injuries', 'Hostages', 'Suicide', 'Claimed',
             'LogFatal', 'LogInj', 'LogHostages']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=.5, ax=ax)
ax.set_title('GRID casualty and indicator correlation heatmap')
plt.tight_layout(); plt.show()


## Summary of findings

A concise analytical report of the key numbers produced from the supplied GRID snapshot.


In [ ]:
total = len(df)
deaths = int(df['Fatalities'].sum())
inj = int(df['Injuries'].sum())
host = int(df['Hostages'].sum())
date_min = df['Date'].min()
date_max = df['Date'].max()
pk = int(df.groupby('Year')['IncidentID'].count().idxmax())

print('=' * 72)
print('1) Overall GRID picture')
print('   Incidents ............: %d' % total)
print('   Total fatalities .....: %d' % deaths)
print('   Total injuries .......: %d' % inj)
print('   Hostages/kidnapped/...: %d' % host)
print('   Avg fatalities ......: %.2f per incident' % (deaths / total))
print('   Suicide share ........: %.1f%%' % (100.0 * df['Suicide'].mean()))
print('   Claimed share ........: %.1f%%' % (100.0 * df['Claimed'].mean()))
print('   Date range ...........: %s to %s' % (date_min.date(), date_max.date()))
print()
print('2) Top countries by incident count')
for c_, v in df['Country'].value_counts().head(8).items():
    dd = int(df.loc[df['Country'] == c_, 'Fatalities'].sum())
    print('   %-28s : %5d incidents, %6d fatalities' % (c_, v, dd))
print()
print('3) Annual trend')
yct = df.groupby('Year')['IncidentID'].count()
print('   Peak year: %d (%d incidents)' % (int(yct.idxmax()), int(yct.max())))
print()
print('4) Primary tactics')
for a_, v in df['PrimaryTactic'].value_counts().head(8).items():
    print('   %-18s : %5d' % (a_, v))
print()
print('5) Primary weapons')
for w_, v in df['PrimaryWeapon'].value_counts().head(8).items():
    print('   %-35s : %5d' % (w_, v))
print()
print('6) Perpetrator coverage')
print('   Distinct perpetrator names: %d' % perps_raw['perp_name'].nunique())
print('   Incidents with no linked perpetrator record: %d' %
      int((df['TerroristGroup'] == 'Unknown / Unclaimed').sum()))
print()
print('7) Source coverage')
print('   Source records: %d' % source_records)
print('   Distinct source-linked incidents: %d' % source_incidents)


## Interactive dashboard

The final cell exports a self-contained interactive HTML dashboard (grid_terrorism_dashboard.html) built with Plotly. It contains KPIs, time trends, country/region rankings, tactic and weapon distributions, perpetrator activity, casualty relationships and an interactive geographic view.

The dashboard includes prominent GRID attribution and the GTTAC/GRID source links required by the supplied Terms of Use and Citation Policy.


In [ ]:
if not HAS_PLOTLY:
    raise RuntimeError('Plotly is required to build the dashboard')

yd = df.groupby('Year').agg(
    Incidents=('IncidentID', 'count'),
    Fatalities=('Fatalities', 'sum'),
    Injuries=('Injuries', 'sum')
).reset_index()

fig1 = px.line(yd, x='Year', y='Incidents', title='GRID incidents per year',
               markers=True, template='plotly_white')
fig1.update_traces(line_width=2.5, marker_size=5)
fig1.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=35))

c15 = df['Country'].value_counts().head(15).sort_values().reset_index()
c15.columns = ['Country', 'Incidents']
fig2 = px.bar(c15, x='Incidents', y='Country', orientation='h',
              title='Top 15 countries by GRID incidents',
              color='Incidents', color_continuous_scale='YlOrRd',
              template='plotly_white')
fig2.update_layout(height=400, margin=dict(l=10, r=20, t=45, b=10), showlegend=False)

tacd = df['PrimaryTactic'].value_counts().reset_index()
tacd.columns = ['PrimaryTactic', 'Incidents']
fig3 = px.bar(tacd, x='PrimaryTactic', y='Incidents',
              title='Incidents by primary tactic category',
              color='PrimaryTactic',
              color_discrete_sequence=px.colors.qualitative.Set2,
              template='plotly_white')
fig3.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=65), showlegend=False)
fig3.update_xaxes(tickangle=25)

regd = df['Region'].value_counts().reset_index()
regd.columns = ['Region', 'Incidents']
fig4 = px.pie(regd, names='Region', values='Incidents',
              title='Share of incidents by GRID region',
              color_discrete_sequence=px.colors.qualitative.Set2,
              template='plotly_white')
fig4.update_layout(height=380, margin=dict(l=15, r=15, t=45, b=15))

grd = (perps_raw.groupby('perp_name')['unique_incident_id']
       .nunique().sort_values().tail(12).reset_index())
grd.columns = ['Perpetrator', 'Incidents']
fig5 = px.bar(grd, x='Incidents', y='Perpetrator', orientation='h',
              title='Top perpetrator names by distinct incidents',
              color='Incidents', color_continuous_scale='Purples',
              template='plotly_white')
fig5.update_layout(height=400, margin=dict(l=10, r=20, t=45, b=10), showlegend=False)

fig6 = px.line(yd, x='Year', y='Fatalities', title='Total fatalities per year',
               markers=True, template='plotly_white',
               color_discrete_sequence=['#a50f15'])
fig6.update_traces(line_width=2.5, marker_size=5)
fig6.update_layout(height=360, margin=dict(l=50, r=20, t=45, b=35))

fig7 = px.scatter(df, x='Injuries', y='Fatalities', color='Region',
                  title='Casualties per incident', size='Injuries',
                  hover_name='Country',
                  hover_data=['PrimaryTactic', 'PrimaryWeapon', 'Suicide'],
                  color_discrete_sequence=px.colors.qualitative.Set2,
                  template='plotly_white')
fig7.update_layout(height=380, margin=dict(l=50, r=20, t=45, b=35), legend_font_size=9)

ca = df.dropna(subset=['Lat', 'Lon']).groupby('Country').agg(
    N=('IncidentID', 'count'), F=('Fatalities', 'sum'),
    lat=('Lat', 'mean'), lon=('Lon', 'mean')
).reset_index()
fig8 = px.scatter_geo(ca, lat='lat', lon='lon', size='N', color='F',
                      hover_name='Country',
                      hover_data={'lat': False, 'lon': False, 'F': True, 'N': True},
                      title='Geographic distribution of GRID incidents',
                      color_continuous_scale='YlOrRd',
                      projection='natural earth', template='plotly_white')
fig8.update_geos(showland=True, landcolor='#e8efe8',
                 subunitcolor='#bbbbbb', countrycolor='#cccccc')

kpi_html = (
    '<div class="section"><div class="kpis">'
    '<div class="kpi"><span>Incidents</span><span class="v">' + format(len(df), ",") + '</span></div>'
    '<div class="kpi"><span>Fatalities</span><span class="v">' + format(deaths, ",") + '</span></div>'
    '<div class="kpi"><span>Injured</span><span class="v">' + format(inj, ",") + '</span></div>'
    '<div class="kpi"><span>Hostages / kidnapped / missing</span><span class="v">' + format(host, ",") + '</span></div>'
    '<div class="kpi"><span>Peak year</span><span class="v">' + str(pk) + '</span></div>'
    '</div></div>'
)

pairs = [(fig1, fig2), (fig3, fig4), (fig5, fig6), (fig7, fig8)]
chart_html = ''
first = True
for a, b in pairs:
    if first:
        ha = a.to_html(full_html=False, include_plotlyjs='inline', default_width='100%')
        first = False
    else:
        ha = a.to_html(full_html=False, include_plotlyjs=False, default_width='100%')
    hb = b.to_html(full_html=False, include_plotlyjs=False, default_width='100%')
    chart_html += '<div class="row">' + ha + hb + '</div>'

HEAD = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>GRID Terrorism Data Dashboard</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{font-family:'Segoe UI',Arial,sans-serif;background:#f4f6f9;color:#1d2939;padding:18px}
h1{text-align:center;margin-bottom:4px;color:#0f1b52;font-size:24px}
p.sub{text-align:center;color:#5b6b7f;font-size:13px;margin-bottom:18px}
.note{max-width:1040px;margin:0 auto 14px;background:#fff8e8;border:1px solid #f2d98a;color:#5b4a16;
  font-size:12px;padding:10px 12px;border-radius:8px;line-height:1.45}
.section{background:#fff;border-radius:12px;box-shadow:0 1px 4px rgba(0,0,0,.08);padding:18px;margin:0 auto 18px;max-width:1180px}
.row{display:flex;flex-wrap:wrap;gap:12px;margin-top:14px}
.row>div{flex:1 1 46%;min-width:330px}
.kpis{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:10px}
.kpi{flex:1 1 150px;background:#eef3fc;border-radius:10px;padding:12px 14px;text-align:center}
.kpi .v{display:block;font-size:20px;font-weight:700;color:#0f1b52;margin-top:3px}
.kpi span{font-size:12px;color:#53606f}
.footer{max-width:1180px;margin:12px auto;color:#667085;font-size:11px;line-height:1.5}
.footer a{color:#3155a4}
@media(max-width:560px){.row>div{min-width:100%}}
</style>
</head>
<body>
<h1>Global Terrorism Dashboard</h1>
<p class="sub">GTTAC Record of Incident Database (GRID) - supplied snapshot</p>
<div class="note">
<strong>Source:</strong> Global Terrorism Trends and Analysis Center (GTTAC), Record of Incident Database (GRID).
GRID is an open-source incident database. This dashboard is an original visualization generated from the supplied GRID snapshot.
</div>
"""
TAIL = """
<div class="footer">
<strong>GRID attribution:</strong> Global Terrorism Trends and Analysis Center (GTTAC), GTTAC Record of Incident Database (GRID).
Data access date: 2026-10-08. Source: <a href="https://www.GTTAC.com" target="_blank">GTTAC</a> /
<a href="https://www.GRIDdata.com" target="_blank">GRIDdata.com</a>.<br>
This dashboard uses original visualizations generated from GRID data and is not a reproduction of a published GTTAC/GRID graphic.
</div>
</body>
</html>
"""
with open('grid_terrorism_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(HEAD + kpi_html + chart_html + TAIL)
print('Dashboard written: grid_terrorism_dashboard.html (%.2f MB)' %
      (os.path.getsize('grid_terrorism_dashboard.html') / 1e6))
print('GRID attribution included on the dashboard.')
